# Speculative Decoding with Interruptible Streaming

In [1]:
import openai
import asyncio

In [2]:
client = openai.AsyncOpenAI(base_url="http://127.0.0.1:5000/v1", api_key="0608da5d28eb10cea2914f3de0f3ddba")

In [ ]:
async def generate_response(prompt, stop_event):
    """Streams response but cancels immediately if stop_event is set."""
    try:
        response = await client.chat.completions.create(
            model="cognitivecomputations_dolphin-2.9-llama3-8b",
            messages=[{"role": "user", "content": prompt}],
            stream=True
        )
        async for chunk in response:
            if stop_event.is_set():  # Stop streaming if interrupted
                print("\n[Interrupted]\n")
                stop_event.clear()  # Reset event for next run
                await response.close()
                return
            if chunk.choices[0].delta.content is None: break

            print(chunk.choices[0].delta.content, end="")
    except asyncio.CancelledError:
        print("\n[Generation Cancelled]\n")

async def prewritten_stream():
    stop_event = asyncio.Event()
    current_task = None
    
    # Simulated input changes
    prompts = [
        "Tell me about the history of space exploration.",
        "Actually, focus on the Apollo missions.",
        "Wait, just tell me about the Apollo 11 moon landing."
    ]
    
    for new_prompt in prompts:
        if current_task and not current_task.done():
            stop_event.set()
            await asyncio.sleep(0.01)
        
        print(f"\n[New Input Detected: \"{new_prompt}\"]\n[Generating Response...]\n")
        current_task = asyncio.create_task(generate_response(new_prompt, stop_event))
        await asyncio.sleep(2)

await prewritten_stream()



[New Input Detected: "Tell me about the history of space exploration."]
[Generating Response...]

Space exploration refers to the investigation of space beyond Earth's atmosphere. Since ancient times, humanity has gazed up at the stars, trying to understand the nature of the universe. The last century has seen incredible developments in space exploration, here is a brief history:

1957: The Soviet Union
[New Input Detected: "Actually, focus on the Apollo missions."]
[Generating Response...]


[Interrupted]

Apollo 1 Mission:

Apollo 1 was the first crewed mission of NASA's Apollo program, which was launched on January 16, 1967. It was planned to test the Command and Service Module (CSM) design and spacecraft systems. Tr
[New Input Detected: "Wait, just tell me about the Apollo 11 moon landing."]
[Generating Response...]


[Interrupted]

The Apollo 11 moon landing was the first successful manned mission to land on the moon, which took place on July 20, 1969. Astronauts Neil Armstrong a

 the moon, while astronaut Michael Collins remained in the command module Columbia orbiting above. Armstrong and Aldrin spent about 2.5 hours on the moon's surface, conducting experiments and taking photographs. Neil Armstrong was the first person to step onto the moon's surface, famously saying, "That's one small step for man, one giant leap for mankind." This historic event marked a significant achievement in human space exploration and was a major milestone in the Space Race between the United States and the Soviet Union.